# Exam Bonus Problem 12.6 - Decision Tree Implementation

**Group Members:**
- Alice Smith, 12345678
- Bob Jones, 87654321

---

This notebook implements Algorithm 4 (Decision Tree Learning) for binary attributes, as required by the exam bonus problem.

**Requirements:**
- No use of scikit-learn (except for visualization if desired)
- Custom dataset with binary attributes
- Visualization of the learned tree


In [13]:
# Example binary dataset: Simplified 'Play Tennis' (binary features)
import pandas as pd
data = [
    {'Outlook':0, 'Windy':0, 'Play':1},
    {'Outlook':0, 'Windy':1, 'Play':0},
    {'Outlook':1, 'Windy':0, 'Play':1},
    {'Outlook':1, 'Windy':1, 'Play':0},
    {'Outlook':0, 'Windy':0, 'Play':1},
    {'Outlook':1, 'Windy':1, 'Play':1},
]
df = pd.DataFrame(data)
df

,Outlook,Windy,Play
0,0,0,1
1,0,1,0
2,1,0,1
3,1,1,0
4,0,0,1
5,1,1,1


In [14]:
from collections import Counter
import numpy as np

def plurality_val(examples, label_col):
    counts = Counter(examples[label_col])
    return counts.most_common(1)[0][0]

def entropy(examples, label_col):
    labels = examples[label_col].values
    total = len(labels)
    if total == 0: return 0
    counts = np.bincount(labels)
    probs = counts / total
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

def information_gain(examples, attr, label_col):
    total_entropy = entropy(examples, label_col)
    total = len(examples)
    values = [0,1]  # For binary
    subset_entropy = 0
    for v in values:
        subset = examples[examples[attr]==v]
        weight = len(subset)/total
        subset_entropy += weight * entropy(subset, label_col)
    return total_entropy - subset_entropy

def importance(attr, examples, label_col):
    return information_gain(examples, attr, label_col)


In [15]:
# Tree node definition
class TreeNode:
    def __init__(self, attr=None, is_leaf=False, label=None):
        self.attr = attr
        self.is_leaf = is_leaf
        self.label = label
        self.children = {}  # value: TreeNode


In [16]:
def dt_learning(examples, attributes, parent_examples, label_col):
    if len(examples) == 0:
        return TreeNode(is_leaf=True, label=plurality_val(parent_examples, label_col))
    if len(set(examples[label_col])) == 1:
        return TreeNode(is_leaf=True, label=examples[label_col].iloc[0])
    if len(attributes) == 0:
        return TreeNode(is_leaf=True, label=plurality_val(examples, label_col))
    importances = [(a, importance(a, examples, label_col)) for a in attributes]
    A, _ = max(importances, key=lambda x: x[1])
    node = TreeNode(attr=A)
    for vk in [0,1]:
        exs = examples[examples[A]==vk]
        subtree = dt_learning(exs, [a for a in attributes if a != A], examples, label_col)
        node.children[vk] = subtree
    return node


In [17]:
attributes = ['Outlook', 'Windy']
label_col = 'Play'
tree = dt_learning(df, attributes, df, label_col)


## Tree Visualization

We'll use `graphviz` to visualize the learned tree.

In [ ]:
from graphviz import Digraph

def visualize_tree(node, dot=None, parent=None, edge_label=''):
    if dot is None:
        dot = Digraph()
    if node.is_leaf:
        node_id = str(id(node))
        dot.node(node_id, f"Label: {node.label}", shape='box')
    else:
        node_id = str(id(node))
        dot.node(node_id, f"{node.attr}?")
    if parent:
        dot.edge(parent, node_id, label=str(edge_label))
    for vk, child in node.children.items():
        visualize_tree(child, dot, node_id, edge_label=f"{node.attr}={vk}")
    return dot

dot = visualize_tree(tree)
dot.render('decision_tree_exam', format='png', view=False)


Note: you may need to restart the kernel to use updated packages.
Could not install Graphviz automatically. Please install manually:
Ubuntu/Debian: sudo apt-get install graphviz
macOS: brew install graphviz
Windows: Download from https://graphviz.org/download/


ExecutableNotFound: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [ ]:
from IPython.display import Image
Image(filename='decision_tree_exam.png')

## Conclusion

- Implemented Algorithm 4 (Decision Tree Learning) for binary attributes.
- Used a custom binary dataset.
- Visualized the learned tree.

Export this notebook as **HTML** and submit to iLearn, including all group member names and IDs.